# Processing the data (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
# !pip install datasets evaluate transformers[sentencepiece]
# !pip install --upgrade pip
# !pip install torch
# !pip install transformers[torch]
#!pip install scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 84.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 95.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [2]:
from datasets import load_dataset
import csv

# 1. Définir les noms de colonnes selon la description du dataset LIAR
col_names = [
    "id", "label_text", "statement", "subject", "speaker", 
    "job_title", "state_info", "party_affiliation", 
    "barely_true_counts", "false_counts", "half_true_counts", 
    "mostly_true_counts", "pants_on_fire_counts", "context"
]

# 2. Charger les fichiers locaux
# Assurez-vous que les fichiers .tsv sont dans votre environnement
raw_datasets = load_dataset(
    "csv", 
    data_files={
        "train": "/home/onyxia/work/Stat_App/Data/train.tsv", 
        "validation": "/home/onyxia/work/Stat_App/Data/valid.tsv", 
        "test": "/home/onyxia/work/Stat_App/Data/test.tsv"
    }, 
    delimiter="\t", 
    column_names=col_names,
    quoting=csv.QUOTE_NONE
)

# 3. Créer un mapping pour les labels (Texte -> Entier)
# LIAR a 6 classes. Le modèle BERT a besoin d'entiers (0, 1, 2...).
label_mapping = {
    'pants-fire': 0, 
    'false': 1, 
    'barely-true': 2, 
    'half-true': 3, 
    'mostly-true': 4, 
    'true': 5
}

def map_labels(example):
    return {'label': label_mapping[example['label_text']]}

# Appliquer la conversion des labels
raw_datasets = raw_datasets.map(map_labels)

print(raw_datasets)

/home/onyxia/work/venv_sa/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 10269 examples [00:00, 130353.60 examples/s]
Generating validation split: 1284 examples [00:00, 57346.70 examples/s]
Generating test split: 1283 examples [00:00, 99877.36 examples/s]
Map: 100%|██████████| 1283/1283 [00:00<00:00, 5852.04 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 10269
    })
    validation: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 1284
    })
    test: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 1283
    })
})


In [3]:
from transformers import AutoTokenizer, DataCollatorWithPadding

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    # On ne passe que "statement" car c'est une classification de phrase unique
    return tokenizer(example["statement"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Supprimer les colonnes inutiles pour ne garder que ce dont le modèle a besoin
# Cela évite les erreurs de type "text input must be str" s'il reste des métadonnées bizarres
cols_to_keep = ["input_ids", "attention_mask", "label"]
tokenized_datasets = tokenized_datasets.remove_columns(
    [c for c in tokenized_datasets["train"].column_names if c not in cols_to_keep]
)

# Formatage pour PyTorch
tokenized_datasets.set_format("torch")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map: 100%|██████████| 1283/1283 [00:00<00:00, 22831.11 examples/s]


# FINE TUNING

In [4]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")

In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

# 1. Charger la métrique "accuracy" (Précision)
# On utilise 'accuracy' car c'est une classification standard
metric = evaluate.load("accuracy")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    # On prend l'index de la plus haute probabilité (argmax) pour trouver la classe prédite (0 à 5)
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 2. Configurer les hyperparamètres d'entraînement
training_args = TrainingArguments(
    output_dir="liar-bert-finetuned", # Dossier où le modèle sera sauvegardé
    eval_strategy="epoch",            # Évaluer le modèle à la fin de chaque époque
    save_strategy="epoch",            # Sauvegarder le checkpoint à la fin de chaque époque
    learning_rate=2e-5,               # Vitesse d'apprentissage recommandée pour BERT
    per_device_train_batch_size=16,   # Taille des lots (baissez à 8 si erreur de mémoire GPU)
    per_device_eval_batch_size=16,
    num_train_epochs=3,               # Nombre de fois que le modèle voit tout le dataset
    weight_decay=0.01,
    load_best_model_at_end=True,      # À la fin, garder la meilleure version (pas forcément la dernière)
)

# 3. Charger le modèle vierge (Pre-trained) avec 6 labels
checkpoint = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=6)

# 4. Initialiser le Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,             # Passe le tokenizer pour qu'il gère le padding final si besoin
    data_collator=data_collator,     # Votre collator défini précédemment
    compute_metrics=compute_metrics, # La fonction définie plus haut
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_7591/1497315562.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
